In [7]:
# Import SparkSession from PySpark
import pyspark
from pyspark.sql import SparkSession

# Create a local SparkSession
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Hw6Spark") \
        .getOrCreate()

# Print the Spark version
print(spark.version)

4.1.1


In [8]:
#Downloading yellow_tripdata_2025-11.parquet
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-12 02:10:30--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.60, 13.35.33.83, 13.35.33.98, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   187MB/s    in 0.4s    

2026-03-12 02:10:30 (187 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [9]:
# Create dataframe from Yellow 2025-11 data
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [10]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [11]:
df = df.repartition(4)

In [12]:
df.write.parquet('yellow_tripdata/2025/11/')

In [13]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [14]:
# Calculate the average size of all Parquet files 
import os

output_path = "yellow_tripdata"   

total_size = 0
file_count = 0

for root, dirs, files in os.walk(output_path):
    for file in files:
        if file.endswith(".parquet"):
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)
            file_count += 1

if file_count > 0:
    avg_size_mb = total_size / file_count / (1024 * 1024)
    print(f"Average size of Parquet files: {avg_size_mb:.2f} MB")
else:
    print("No parquet files found.")


Average size of Parquet files: 25.33 MB


In [15]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [16]:
# Prepare for use of sql functions
from pyspark.sql import functions as F

In [17]:
# Create new fields as date-only, converted from datetime fields
# to convert timestamp -> date using F.to_date()
df_updated = df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.tpep_pickup_datetime))

In [18]:
df_updated.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|  

In [19]:
# Prepare temp table from updated schema & data
df_updated.registerTempTable('yellow_nov_data')

/workspaces/DataTalks-Data-Engineering-Zoomcamp/batch/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [20]:
# Verify sample SQL statement with new temp table
spark.sql("""
SELECT * FROM yellow_nov_data LIMIT 10;
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_date|dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|  

In [21]:
# Calculate total number of trips on 2025-11-15
spark.sql("""
    SELECT COUNT(*) AS total_trips 
    FROM yellow_nov_data
    WHERE pickup_date = '2025-11-15'
""").show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+



In [22]:
#Calculate the longest trip in hours
spark.sql("""
SELECT ROUND(MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600), 1) 
AS longest_trip_hour
FROM yellow_nov_data
""").show()

+-----------------+
|longest_trip_hour|
+-----------------+
|             90.6|
+-----------------+



In [23]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-12 02:11:13--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.60, 13.35.33.10, 13.35.33.98, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-12 02:11:14 (802 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [24]:
# Load CSV
taxi_zones_df = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

In [25]:
# Register DataFrame as a temp table
taxi_zones_df.createOrReplaceTempView("taxi_zone_lookup")

In [26]:
# Use Spark SQL to find the least frequent pickup zone
spark.sql("""
WITH zone_counts AS (
    SELECT t.Zone, COUNT(*) AS trip_count
    FROM yellow_nov_data y
    JOIN taxi_zone_lookup t
      ON y.PULocationID = t.LocationID
    GROUP BY t.Zone
)
SELECT Zone, trip_count
FROM zone_counts
WHERE trip_count = (SELECT MIN(trip_count) FROM zone_counts)
""").show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|Eltingville/Annad...|         1|
|       Arden Heights|         1|
+--------------------+----------+

